# 🌳 Week 5 - Lab 3: Decision Trees and Random Forests

## Session 23: Classification Models II - Building Powerful Ensemble Models

---

### 📚 Learning Objectives

By the end of this lab, you will be able to:

1. **Understand** how Decision Trees make predictions through recursive splitting
2. **Build and visualize** Decision Tree classifiers
3. **Understand** the concept of ensemble learning and Random Forests
4. **Implement** Random Forest classifiers and understand their advantages
5. **Extract and interpret** feature importance from tree-based models
6. **Tune hyperparameters** to optimize model performance
7. **Compare** Decision Trees vs Random Forests

---

### 🎯 Today's Focus: From Single Trees to Forest Power!

We'll explore how Decision Trees work (intuitive, interpretable) and then see how combining many trees into a Random Forest creates a more powerful and robust model.

**Key Concepts:**
- Decision Trees: "If-then" rules learned from data
- Random Forests: "Wisdom of the crowd" - many trees voting together
- Feature Importance: Which features matter most for predictions?

---

## Part 1: Setup and Data Loading

Let's import all necessary libraries and load our dataset.

In [ ]:
# ============================================================
# CELL 1: Import Required Libraries
# ============================================================
# Run this cell first to import all necessary packages

# Core data manipulation libraries
import numpy as np                    # Numerical computing
import pandas as pd                   # Data manipulation and analysis

# Visualization libraries
import matplotlib.pyplot as plt       # Basic plotting
import seaborn as sns                 # Statistical visualization

# Scikit-learn: Machine Learning library
from sklearn.datasets import load_wine           # Wine dataset (multi-class)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler

# Tree-based models
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# Evaluation metrics
from sklearn.metrics import (accuracy_score, classification_report, 
                             confusion_matrix, ConfusionMatrixDisplay)

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✅ All libraries imported successfully!")
print("\n📦 Key packages we'll use:")
print("   • DecisionTreeClassifier - Single decision tree model")
print("   • RandomForestClassifier - Ensemble of decision trees")
print("   • plot_tree - Visualize decision tree structure")
print("   • GridSearchCV - Hyperparameter tuning")

In [ ]:
# ============================================================
# CELL 2: Load the Wine Dataset
# ============================================================
# We'll use the Wine dataset - a classic multi-class classification problem
# Goal: Classify wines into 3 categories based on chemical properties

# Load the wine dataset from sklearn
wine = load_wine()

# Create a DataFrame for easier manipulation
# wine.data contains the features, wine.feature_names has column names
df = pd.DataFrame(wine.data, columns=wine.feature_names)

# Add the target variable (wine class: 0, 1, or 2)
df['target'] = wine.target

# Map target numbers to meaningful names
target_names = {0: 'Class_0', 1: 'Class_1', 2: 'Class_2'}
df['wine_class'] = df['target'].map(target_names)

# Display dataset information
print("🍷 WINE DATASET LOADED")
print("=" * 50)
print(f"\n📊 Dataset Shape: {df.shape[0]} samples × {df.shape[1]} columns")
print(f"\n🎯 Target Classes: {wine.target_names}")
print(f"\n📋 Features ({len(wine.feature_names)}):")
for i, name in enumerate(wine.feature_names, 1):
    print(f"   {i:2d}. {name}")

print("\n" + "=" * 50)
print("\n🔍 First 5 rows of the dataset:")
df.head()

In [ ]:
# ============================================================
# CELL 3: Explore the Target Distribution
# ============================================================
# Let's see how many samples we have in each wine class

# Count samples per class
class_counts = df['wine_class'].value_counts().sort_index()

# Create a bar plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='black')
axes[0].set_xlabel('Wine Class', fontsize=12)
axes[0].set_ylabel('Number of Samples', fontsize=12)
axes[0].set_title('Distribution of Wine Classes', fontsize=14, fontweight='bold')

# Add count labels on bars
for i, (idx, val) in enumerate(zip(class_counts.index, class_counts.values)):
    axes[0].text(i, val + 1, str(val), ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
            colors=colors, explode=[0.02, 0.02, 0.02], shadow=True)
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Class Distribution Summary:")
print(class_counts.to_string())
print(f"\n✅ Dataset is relatively balanced - good for classification!")

In [ ]:
# ============================================================
# CELL 4: Prepare Data for Modeling
# ============================================================
# Split our data into features (X) and target (y)
# Then split into training and testing sets

# Separate features and target
X = df[wine.feature_names]  # All feature columns
y = df['target']            # Target column (0, 1, or 2)

# Split into training (80%) and testing (20%) sets
# stratify=y ensures each class is proportionally represented in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    random_state=42,    # For reproducibility
    stratify=y          # Maintain class proportions
)

print("📊 DATA SPLIT COMPLETE")
print("=" * 50)
print(f"\n🔹 Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"🔹 Testing set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\n🔹 Number of features: {X_train.shape[1]}")

print("\n📈 Class distribution in splits:")
print(f"   Training: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"   Testing:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

---

## Part 2: Understanding Decision Trees

### How Decision Trees Work:

Decision Trees learn **if-then rules** from data by recursively splitting the dataset:

1. **Start at the root** with all training data
2. **Find the best feature and threshold** to split the data (maximize information gain)
3. **Create child nodes** for each split
4. **Repeat** until stopping criteria is met (max depth, min samples, etc.)
5. **Assign class labels** to leaf nodes based on majority class

**Key Concepts:**
- **Gini Impurity**: Measures how "mixed" the classes are at a node (0 = pure, 0.5 = maximum impurity for binary)
- **Information Gain**: Reduction in impurity after a split

In [ ]:
# ============================================================
# CELL 5: Train a Basic Decision Tree
# ============================================================
# Let's create our first Decision Tree classifier

# Create a Decision Tree with default parameters
# Note: No max_depth means the tree will grow until all leaves are pure
dt_basic = DecisionTreeClassifier(
    random_state=42,      # For reproducibility
    criterion='gini'      # Use Gini impurity for splits (default)
)

# Train the model on our training data
dt_basic.fit(X_train, y_train)

# Make predictions on training and test sets
y_train_pred = dt_basic.predict(X_train)
y_test_pred = dt_basic.predict(X_test)

# Calculate accuracy scores
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("🌳 BASIC DECISION TREE TRAINED")
print("=" * 50)
print(f"\n📊 Tree Properties:")
print(f"   • Depth: {dt_basic.get_depth()}")
print(f"   • Number of leaves: {dt_basic.get_n_leaves()}")
print(f"   • Number of features used: {dt_basic.n_features_in_}")

print(f"\n🎯 Accuracy Results:")
print(f"   • Training Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"   • Testing Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")

# Check for overfitting
if train_acc == 1.0 and test_acc < 0.95:
    print("\n⚠️  Warning: 100% training accuracy with lower test accuracy")
    print("   This suggests OVERFITTING - the tree memorized the training data!")

In [ ]:
# ============================================================
# CELL 6: Visualize the Decision Tree
# ============================================================
# One of the biggest advantages of Decision Trees: interpretability!
# We can visualize exactly how the model makes decisions

# Create a figure for the tree visualization
plt.figure(figsize=(20, 12))

# Plot the decision tree
plot_tree(
    dt_basic,                          # Our trained tree
    feature_names=wine.feature_names,  # Feature names for readability
    class_names=wine.target_names,     # Class names
    filled=True,                       # Color nodes by class
    rounded=True,                      # Rounded corners
    fontsize=8,                        # Font size
    proportion=True                    # Show proportions instead of counts
)

plt.title('Decision Tree Visualization (Full Tree - May Be Complex!)', 
          fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📖 How to Read the Tree:")
print("   • Each box shows: feature ≤ threshold")
print("   • 'gini': Impurity measure (0 = pure node)")
print("   • 'samples': Percentage of training samples at this node")
print("   • 'value': Distribution of classes [Class_0, Class_1, Class_2]")
print("   • 'class': Predicted class if this were a leaf node")
print("   • Colors: Darker = more pure (confident prediction)")

In [ ]:
# ============================================================
# CELL 7: Create a Simpler Tree (Limit Depth)
# ============================================================
# Deep trees tend to overfit. Let's create a shallower tree
# that's easier to interpret and may generalize better

# Create a Decision Tree with limited depth
dt_simple = DecisionTreeClassifier(
    max_depth=3,          # Limit tree to 3 levels
    random_state=42,
    criterion='gini'
)

# Train the model
dt_simple.fit(X_train, y_train)

# Make predictions
y_train_pred_simple = dt_simple.predict(X_train)
y_test_pred_simple = dt_simple.predict(X_test)

# Calculate accuracy
train_acc_simple = accuracy_score(y_train, y_train_pred_simple)
test_acc_simple = accuracy_score(y_test, y_test_pred_simple)

print("🌳 SIMPLE DECISION TREE (max_depth=3)")
print("=" * 50)
print(f"\n📊 Tree Properties:")
print(f"   • Depth: {dt_simple.get_depth()}")
print(f"   • Number of leaves: {dt_simple.get_n_leaves()}")

print(f"\n🎯 Accuracy Comparison:")
print(f"                    Full Tree    Simple Tree")
print(f"   Training:        {train_acc*100:6.2f}%       {train_acc_simple*100:6.2f}%")
print(f"   Testing:         {test_acc*100:6.2f}%       {test_acc_simple*100:6.2f}%")

# Visualize the simpler tree
plt.figure(figsize=(16, 8))
plot_tree(
    dt_simple,
    feature_names=wine.feature_names,
    class_names=wine.target_names,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Simple Decision Tree (max_depth=3) - Easier to Interpret!', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 8: Find Optimal Tree Depth
# ============================================================
# Let's systematically test different tree depths to find the best one

# Test depths from 1 to 15
depths = range(1, 16)
train_scores = []
test_scores = []

for depth in depths:
    # Create and train tree with this depth
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    
    # Record scores
    train_scores.append(accuracy_score(y_train, dt.predict(X_train)))
    test_scores.append(accuracy_score(y_test, dt.predict(X_test)))

# Find best depth based on test accuracy
best_depth = depths[np.argmax(test_scores)]
best_test_acc = max(test_scores)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores, 'b-o', label='Training Accuracy', linewidth=2, markersize=8)
plt.plot(depths, test_scores, 'r-s', label='Testing Accuracy', linewidth=2, markersize=8)

# Mark the best depth
plt.axvline(x=best_depth, color='green', linestyle='--', label=f'Best Depth = {best_depth}')

# Shade overfitting region
plt.fill_between(depths, train_scores, test_scores, alpha=0.2, color='orange',
                 label='Overfitting Gap')

plt.xlabel('Tree Depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Finding the Optimal Tree Depth', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.xticks(depths)
plt.ylim(0.7, 1.05)
plt.tight_layout()
plt.show()

print(f"\n🎯 Optimal Tree Depth: {best_depth}")
print(f"   Best Test Accuracy: {best_test_acc*100:.2f}%")
print("\n💡 Observation: Notice how training accuracy keeps increasing with depth,")
print("   but test accuracy plateaus or decreases - this is OVERFITTING!")

---

## Part 3: Random Forests - The Power of Ensemble Learning

### The Problem with Single Decision Trees:
- **High Variance**: Small changes in data can lead to very different trees
- **Overfitting**: Deep trees memorize training data

### The Solution: Random Forests! 🌲🌲🌲

A Random Forest is an **ensemble** of many Decision Trees that work together:

1. **Bootstrap Sampling**: Each tree trains on a random subset of data (with replacement)
2. **Feature Randomization**: Each split considers only a random subset of features
3. **Voting**: For classification, each tree votes and the majority wins

**Result**: More robust, less overfitting, better generalization!

In [ ]:
# ============================================================
# CELL 9: Train a Random Forest
# ============================================================
# Let's create a Random Forest and see how it compares to a single tree

# Create a Random Forest classifier
rf = RandomForestClassifier(
    n_estimators=100,     # Number of trees in the forest
    max_depth=None,       # Let trees grow fully
    min_samples_split=2,  # Minimum samples to split a node
    min_samples_leaf=1,   # Minimum samples in a leaf
    random_state=42,      # For reproducibility
    n_jobs=-1             # Use all CPU cores for parallel training
)

# Train the Random Forest
print("🌲 Training Random Forest with 100 trees...")
rf.fit(X_train, y_train)
print("✅ Training complete!")

# Make predictions
y_train_pred_rf = rf.predict(X_train)
y_test_pred_rf = rf.predict(X_test)

# Calculate accuracy
train_acc_rf = accuracy_score(y_train, y_train_pred_rf)
test_acc_rf = accuracy_score(y_test, y_test_pred_rf)

print("\n🌲 RANDOM FOREST RESULTS")
print("=" * 50)
print(f"\n📊 Forest Properties:")
print(f"   • Number of trees: {rf.n_estimators}")
print(f"   • Features per tree: {rf.n_features_in_}")

print(f"\n🎯 Accuracy:")
print(f"   • Training Accuracy: {train_acc_rf:.4f} ({train_acc_rf*100:.2f}%)")
print(f"   • Testing Accuracy:  {test_acc_rf:.4f} ({test_acc_rf*100:.2f}%)")

In [ ]:
# ============================================================
# CELL 10: Compare Single Tree vs Random Forest
# ============================================================
# Let's visually compare the performance of different models

# Collect all results
models = ['Decision Tree\n(Full)', 'Decision Tree\n(Depth=3)', 'Random Forest\n(100 trees)']
train_accs = [train_acc, train_acc_simple, train_acc_rf]
test_accs = [test_acc, test_acc_simple, test_acc_rf]

# Create comparison bar chart
x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, train_accs, width, label='Training', color='#3498db', edgecolor='black')
bars2 = ax.bar(x + width/2, test_accs, width, label='Testing', color='#e74c3c', edgecolor='black')

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{height*100:.1f}%', ha='center', va='bottom', fontsize=10)

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{height*100:.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Comparison: Decision Trees vs Random Forest', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0.8, 1.08)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Observations:")
print("   • Full Decision Tree: Perfect training, may overfit")
print("   • Simple Decision Tree: Less overfitting, but may underfit")
print("   • Random Forest: Best of both worlds - high accuracy, good generalization!")

In [ ]:
# ============================================================
# CELL 11: Detailed Evaluation - Random Forest
# ============================================================
# Let's look at the confusion matrix and classification report

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix for Decision Tree
cm_dt = confusion_matrix(y_test, y_test_pred)
ConfusionMatrixDisplay(cm_dt, display_labels=wine.target_names).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Decision Tree (Full)', fontsize=12, fontweight='bold')

# Confusion Matrix for Random Forest
cm_rf = confusion_matrix(y_test, y_test_pred_rf)
ConfusionMatrixDisplay(cm_rf, display_labels=wine.target_names).plot(ax=axes[1], cmap='Greens')
axes[1].set_title('Random Forest', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Print detailed classification reports
print("\n" + "="*60)
print("📊 RANDOM FOREST - DETAILED CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_test_pred_rf, target_names=wine.target_names))

---

## Part 4: Feature Importance

One of the most powerful aspects of tree-based models is **feature importance**!

Feature importance tells us which features are most useful for making predictions.

### How is it calculated?
- For each feature, measure how much it reduces impurity (Gini) across all trees
- Features that create better splits have higher importance
- Importance values sum to 1.0

In [ ]:
# ============================================================
# CELL 12: Extract and Visualize Feature Importance
# ============================================================
# Random Forests provide a natural way to rank feature importance

# Get feature importances from the Random Forest
importances = rf.feature_importances_

# Create a DataFrame for easy manipulation
feature_importance_df = pd.DataFrame({
    'Feature': wine.feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=True)  # Sort for horizontal bar plot

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Horizontal bar plot
colors = plt.cm.RdYlGn(feature_importance_df['Importance'] / feature_importance_df['Importance'].max())
axes[0].barh(feature_importance_df['Feature'], feature_importance_df['Importance'], 
             color=colors, edgecolor='black')
axes[0].set_xlabel('Importance Score', fontsize=12)
axes[0].set_title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Pie chart of top 5 features
top5 = feature_importance_df.tail(5)
others = feature_importance_df.head(len(feature_importance_df)-5)['Importance'].sum()

pie_data = list(top5['Importance']) + [others]
pie_labels = list(top5['Feature']) + ['Others']

axes[1].pie(pie_data, labels=pie_labels, autopct='%1.1f%%', 
            colors=plt.cm.Set3.colors[:6], explode=[0.05]*5 + [0])
axes[1].set_title('Top 5 Features Contribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Print top 5 features
print("\n🏆 TOP 5 MOST IMPORTANT FEATURES:")
print("=" * 50)
for i, row in feature_importance_df.tail(5).iloc[::-1].iterrows():
    bar = '█' * int(row['Importance'] * 40)
    print(f"   {row['Feature']:25s} {row['Importance']:.4f} {bar}")

In [ ]:
# ============================================================
# CELL 13: Compare Feature Importance - Decision Tree vs Random Forest
# ============================================================
# Let's see if both models agree on important features

# Get Decision Tree feature importance
dt_importances = dt_basic.feature_importances_

# Create comparison DataFrame
importance_comparison = pd.DataFrame({
    'Feature': wine.feature_names,
    'Decision Tree': dt_importances,
    'Random Forest': importances
})

# Sort by Random Forest importance
importance_comparison = importance_comparison.sort_values('Random Forest', ascending=True)

# Create comparison plot
fig, ax = plt.subplots(figsize=(12, 8))

y_pos = np.arange(len(wine.feature_names))
width = 0.35

ax.barh(y_pos - width/2, importance_comparison['Decision Tree'], width, 
        label='Decision Tree', color='#3498db', edgecolor='black', alpha=0.8)
ax.barh(y_pos + width/2, importance_comparison['Random Forest'], width, 
        label='Random Forest', color='#27ae60', edgecolor='black', alpha=0.8)

ax.set_yticks(y_pos)
ax.set_yticklabels(importance_comparison['Feature'])
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Feature Importance Comparison: Decision Tree vs Random Forest', 
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print("   • Decision Trees may over-rely on a few features")
print("   • Random Forests provide more balanced importance scores")
print("   • RF importance is more stable due to averaging across many trees")

---

## Part 5: Hyperparameter Tuning

Let's use **GridSearchCV** to find the optimal hyperparameters for our Random Forest.

### Key Hyperparameters:
- **n_estimators**: Number of trees (more = better but slower)
- **max_depth**: Maximum depth of each tree
- **min_samples_split**: Minimum samples required to split a node
- **min_samples_leaf**: Minimum samples required in a leaf node

In [ ]:
# ============================================================
# CELL 14: Hyperparameter Tuning with GridSearchCV
# ============================================================
# Find the best combination of hyperparameters

# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200],           # Number of trees
    'max_depth': [3, 5, 10, None],            # Maximum depth
    'min_samples_split': [2, 5, 10],          # Min samples to split
    'min_samples_leaf': [1, 2, 4]             # Min samples in leaf
}

# Create a base Random Forest
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

# Create GridSearchCV object
# cv=5 means 5-fold cross-validation
print("🔍 Starting GridSearchCV (this may take a minute)...")
print(f"   Testing {3*4*3*3} = {3*4*3*3} parameter combinations with 5-fold CV")
print(f"   Total fits: {3*4*3*3*5} = {3*4*3*3*5}")

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,                    # 5-fold cross-validation
    scoring='accuracy',      # Optimize for accuracy
    n_jobs=-1,               # Use all CPU cores
    verbose=1                # Show progress
)

# Fit the grid search
grid_search.fit(X_train, y_train)

print("\n✅ GridSearchCV Complete!")
print("\n" + "="*60)
print("🏆 BEST PARAMETERS FOUND:")
print("="*60)
for param, value in grid_search.best_params_.items():
    print(f"   • {param}: {value}")

print(f"\n🎯 Best Cross-Validation Score: {grid_search.best_score_:.4f} ({grid_search.best_score_*100:.2f}%)")

In [ ]:
# ============================================================
# CELL 15: Evaluate the Best Model
# ============================================================
# Use the best model from GridSearchCV

# Get the best estimator
best_rf = grid_search.best_estimator_

# Make predictions
y_test_pred_best = best_rf.predict(X_test)
y_train_pred_best = best_rf.predict(X_train)

# Calculate accuracy
train_acc_best = accuracy_score(y_train, y_train_pred_best)
test_acc_best = accuracy_score(y_test, y_test_pred_best)

print("🏆 OPTIMIZED RANDOM FOREST RESULTS")
print("=" * 60)
print(f"\n🎯 Performance:")
print(f"   • Training Accuracy: {train_acc_best:.4f} ({train_acc_best*100:.2f}%)")
print(f"   • Testing Accuracy:  {test_acc_best:.4f} ({test_acc_best*100:.2f}%)")

# Compare with default RF
print(f"\n📊 Improvement over default Random Forest:")
improvement = (test_acc_best - test_acc_rf) * 100
if improvement > 0:
    print(f"   Test accuracy improved by {improvement:.2f}%")
elif improvement < 0:
    print(f"   Test accuracy decreased by {abs(improvement):.2f}% (default was already good!)")
else:
    print(f"   Same performance as default")

# Confusion Matrix
plt.figure(figsize=(8, 6))
cm_best = confusion_matrix(y_test, y_test_pred_best)
ConfusionMatrixDisplay(cm_best, display_labels=wine.target_names).plot(cmap='RdYlGn')
plt.title('Optimized Random Forest - Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📋 Classification Report:")
print(classification_report(y_test, y_test_pred_best, target_names=wine.target_names))

---

## Part 6: Final Model Summary

Let's create a comprehensive summary comparing all our models.

In [ ]:
# ============================================================
# CELL 16: Final Model Comparison Summary
# ============================================================

# Create summary DataFrame
summary_data = {
    'Model': ['Decision Tree (Full)', 'Decision Tree (Depth=3)', 
              'Random Forest (Default)', 'Random Forest (Optimized)'],
    'Training Accuracy': [train_acc, train_acc_simple, train_acc_rf, train_acc_best],
    'Testing Accuracy': [test_acc, test_acc_simple, test_acc_rf, test_acc_best],
    'Overfitting Gap': [train_acc - test_acc, train_acc_simple - test_acc_simple,
                        train_acc_rf - test_acc_rf, train_acc_best - test_acc_best]
}

summary_df = pd.DataFrame(summary_data)
summary_df['Training Accuracy'] = summary_df['Training Accuracy'].apply(lambda x: f"{x*100:.2f}%")
summary_df['Testing Accuracy'] = summary_df['Testing Accuracy'].apply(lambda x: f"{x*100:.2f}%")
summary_df['Overfitting Gap'] = summary_df['Overfitting Gap'].apply(lambda x: f"{x*100:.2f}%")

print("📊 FINAL MODEL COMPARISON")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

# Create visualization
fig, ax = plt.subplots(figsize=(12, 6))

models = ['DT (Full)', 'DT (Depth=3)', 'RF (Default)', 'RF (Optimized)']
train_accs_all = [train_acc, train_acc_simple, train_acc_rf, train_acc_best]
test_accs_all = [test_acc, test_acc_simple, test_acc_rf, test_acc_best]

x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, train_accs_all, width, label='Training', color='#3498db', edgecolor='black')
bars2 = ax.bar(x + width/2, test_accs_all, width, label='Testing', color='#e74c3c', edgecolor='black')

# Add value labels
for bar in bars1 + bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
            f'{height*100:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Complete Model Comparison: Decision Trees vs Random Forests', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend(loc='lower right')
ax.set_ylim(0.85, 1.05)
ax.grid(axis='y', alpha=0.3)

# Add "Best" annotation
best_idx = np.argmax(test_accs_all)
ax.annotate('⭐ Best', xy=(best_idx + width/2, test_accs_all[best_idx] + 0.02),
            fontsize=12, ha='center', fontweight='bold', color='green')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 17: Key Concepts Summary
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    🎓 KEY CONCEPTS SUMMARY                                   ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  🌳 DECISION TREES                                                          ║
║  ─────────────────                                                          ║
║  • Learn if-then rules by recursively splitting data                        ║
║  • Use Gini impurity or Entropy to find best splits                         ║
║  • Highly interpretable - can visualize the entire model                    ║
║  • Prone to overfitting if not pruned (controlled with max_depth)           ║
║                                                                              ║
║  🌲🌲🌲 RANDOM FORESTS                                                       ║
║  ────────────────────                                                        ║
║  • Ensemble of many Decision Trees working together                         ║
║  • Uses Bootstrap sampling + Feature randomization                          ║
║  • More robust, less overfitting than single trees                          ║
║  • Provides feature importance rankings                                     ║
║                                                                              ║
║  📊 FEATURE IMPORTANCE                                                       ║
║  ─────────────────────                                                       ║
║  • Measures how much each feature contributes to predictions                ║
║  • Based on impurity reduction across all trees/splits                      ║
║  • Useful for feature selection and model interpretation                    ║
║                                                                              ║
║  ⚙️ KEY HYPERPARAMETERS                                                     ║
║  ─────────────────────                                                       ║
║  • n_estimators: Number of trees (more = better but slower)                 ║
║  • max_depth: Controls tree depth (prevents overfitting)                    ║
║  • min_samples_split/leaf: Controls when to stop splitting                  ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

---

## 🎯 Practice Exercises

Try these exercises to reinforce your learning!

In [ ]:
# ============================================================
# EXERCISE 1: Try Different Criterion
# ============================================================
# Decision Trees can use 'gini' or 'entropy' for splitting
# Train two trees with different criteria and compare

# YOUR CODE HERE:
# dt_gini = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
# dt_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
# ... train and compare ...

print("Exercise 1: Compare Gini vs Entropy criterion")
print("Hint: Create two trees with different criterion values")

In [ ]:
# ============================================================
# EXERCISE 2: Effect of Number of Trees
# ============================================================
# How does the number of trees affect Random Forest performance?
# Test n_estimators from 10 to 500 and plot the results

# YOUR CODE HERE:
# n_trees_list = [10, 25, 50, 100, 200, 300, 500]
# ... test each and plot accuracy vs n_estimators ...

print("Exercise 2: Plot accuracy vs number of trees")
print("Hint: Loop through different n_estimators values")

In [ ]:
# ============================================================
# EXERCISE 3: Feature Selection Based on Importance
# ============================================================
# Train a model using only the top 5 most important features
# Does the performance change significantly?

# YOUR CODE HERE:
# top_5_features = feature_importance_df.tail(5)['Feature'].tolist()
# X_train_top5 = X_train[top_5_features]
# ... train and evaluate ...

print("Exercise 3: Train with only top 5 features")
print("Hint: Select columns based on feature importance ranking")

---

## 🏁 Lab Complete!

### What You Learned Today:

1. ✅ How Decision Trees make predictions through recursive splitting
2. ✅ How to visualize and interpret Decision Trees
3. ✅ The concept of ensemble learning with Random Forests
4. ✅ How to extract and interpret feature importance
5. ✅ How to tune hyperparameters using GridSearchCV
6. ✅ When to use Decision Trees vs Random Forests

### Next Steps:
- Try these models on your own datasets
- Explore other ensemble methods (Gradient Boosting, XGBoost)
- Practice feature selection based on importance

---

**Great work! 🎉**